## DEPENDENCIAS

In [ ]:
!pip install datasets transformers torchaudio librosa jiwer accelerate -U

### IMPORTACIÓN DE DEPENDENCIAS

In [2]:
import os
import json
import torch
import pandas as pd
from datasets import Dataset, DatasetDict, Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

### Entrenamiento

In [ ]:
# RUTAS DE KAGGLE 
dataset_name = 'dataset-asr-mam-usac'
base_path = f'/kaggle/input/datasets/cdjg10/{dataset_name}'
csv_path = os.path.join(base_path, 'metadata.csv')
audio_dir = os.path.join(base_path, '03_processed')

# OUTPUT
output_dir_modelo = '/kaggle/working/modelo_final'

# SPLIT DE DATOS 90% ENTRENAMIENTO
df = pd.read_csv(csv_path)
df['audio'] = df['file_name'].apply(lambda x: os.path.join(audio_dir, x))
hg_dataset = Dataset.from_pandas(df)
dataset_split = hg_dataset.train_test_split(test_size=0.1, seed=42)
mam_dataset = DatasetDict({"train": dataset_split["train"], "test": dataset_split["test"]})
mam_dataset = mam_dataset.cast_column("audio", Audio(sampling_rate=16000))

# CONSTRUCCION DE VOCABULARIO
def extract_all_chars(batch):
    all_text = " ".join(batch["transcription"])
    return {"vocab": [list(set(all_text))], "all_text": [all_text]}

vocabs = mam_dataset.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=mam_dataset.column_names["train"])
vocab_list = list(set(vocabs["train"]["vocab"][0]) | set(vocabs["test"]["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict["|"] = vocab_dict.get(" ", len(vocab_dict))
if " " in vocab_dict: del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

# SE GUARDA EL VOCABULARIO
with open('/kaggle/working/vocab.json', 'w', encoding='utf-8') as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)

# INICIALIZACION DE TOKENIZADOR Y DEL PROCESADOR
tokenizer = Wav2Vec2CTCTokenizer("/kaggle/working/vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# SE PASAN DE AUDIOS A TENSORES
print("Procesando los audios...")
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = processor(text=batch["transcription"]).input_ids
    return batch

mam_dataset = mam_dataset.map(prepare_dataset, remove_columns=mam_dataset.column_names["train"], num_proc=2)

# COLLATOR
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# RED NEURONAL BASE wav2vec2-xls-r-300m
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)
model.freeze_feature_encoder() 

# SE APLICA EARLY STOPPING
training_args = TrainingArguments(
    output_dir=output_dir_modelo, 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    num_train_epochs=50, 
    fp16=True,
    save_steps=50, 
    eval_steps=50,
    logging_steps=25,
    learning_rate=3e-4,
    warmup_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True, 
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=mam_dataset["train"],
    eval_dataset=mam_dataset["test"],
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)], 
)

print("Iniciando proceso...")
trainer.train()

# SE SELECCIONA EL MEJOR MODELO
trainer.save_model(output_dir_modelo)
processor.save_pretrained(output_dir_modelo)